In [0]:
from pyspark.sql import functions as F, Window as W
import time
 
CATALOG = "airline_analytics"
BRONZE_HIST = f"{CATALOG}.bronze.flights_historical"
BRONZE_RECENT = f"{CATALOG}.bronze.flights_recent"
SILVER = f"{CATALOG}.silver.flights"
QUARANTINE = f"{CATALOG}.silver.flights_quarantine"
PILOT_YEARS = {"historical": [2004, 2006, 2007, 2008],
    "recent": [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,2020, 2021, 2022, 2023, 2024, 2025, 2026],
}

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

In [0]:
def _clean_str(colname):
    """String columns: trim, empty string -> NULL."""
    s = F.trim(F.col(colname).cast("string"))
    return F.when(s == "", None).otherwise(s)
 
 
def num_int(colname):
    """INT (historical) or DOUBLE (recent) -> INT."""
    return F.col(colname).cast("int")
 
 
def num_dbl(colname):
    return F.col(colname).cast("double")
 
 
def to_bool_01(colname):
    """Cancelled/Diverted: INT 0/1 in historical, DOUBLE 0.0/1.0 in recent."""
    return F.coalesce(F.col(colname).cast("int") == 1, F.lit(False))
 
 
def opt(df, colname, expr_fn, dtype="int"):
    """Use a column if the source has it, otherwise emit typed NULLs. Guards the BTS-only
    columns, which the profiling run didn't explicitly confirm."""
    return expr_fn(colname) if colname in df.columns else F.lit(None).cast(dtype)

def hhmm(colname):
    """
    Normalise an hhmm time field.
    2400 and 0 both mean midnight in this data; both are normalised to 0.
    Anything that doesn't decompose into a valid hour/minute becomes NULL.
    """
    raw = num_int(colname)
    v = F.when(raw == 2400, F.lit(0)).otherwise(raw)
    hour, minute = F.floor(v / 100), v % 100
    return F.when((v >= 0) & (v <= 2359) & (hour <= 23) & (minute <= 59), v)
 
 
def hhmm_to_hour(col_expr):
    return F.floor(col_expr / 100).cast("int")
 
 
def local_timestamp(date_col, hhmm_col):
    """flight_date + hhmm -> local wall-clock TIMESTAMP. See the README limitation note."""
    padded = F.lpad(hhmm_col.cast("string"), 4, "0")
    return F.to_timestamp(
        F.concat(
            date_col.cast("string"), F.lit(" "),
            F.substring(padded, 1, 2), F.lit(":"), F.substring(padded, 3, 2),
        ),
        "yyyy-MM-dd HH:mm",
    )
def clean_tail(colname):
    """Registration numbers. The historical source uses placeholder junk for unknowns."""
    t = F.upper(_clean_str(colname))
    junk = ["0", "00", "000000", "UNKNOW", "UNKNOWN", "-", "NONE"]
    return F.when(t.isin(junk) | (F.length(t) < 3), None).otherwise(t)
 
 
CANCELLATION_REASONS = {"A": "Carrier", "B": "Weather", "C": "National Air System", "D": "Security",
}
 
 
def cancellation_reason(code_col):
    out = F.lit(None).cast("string")
    for code, label in CANCELLATION_REASONS.items():
        out = F.when(code_col == code, F.lit(label)).otherwise(out)
    return out
 
 
def metadata_col(df, name, fallback):
    """Bronze metadata columns may not exist on both tables — degrade gracefully."""
    return F.col(name) if name in df.columns else fallback
 

In [0]:
SILVER_COLUMNS = [
    # identity + date
    "flight_key", "flight_date", "year", "quarter", "month", "day_of_month", "day_of_week",
    # carrier / route
    "carrier_code", "tail_number", "flight_number", "origin", "dest",
    # times (hhmm ints, local to the airport)
    "crs_dep_time_hhmm", "dep_time_hhmm", "crs_arr_time_hhmm", "arr_time_hhmm",
    "crs_dep_hour", "scheduled_departure_local_ts",
    # delay measures
    "dep_delay_min", "arr_delay_min", "dep_delay_min_pos", "arr_delay_min_pos",
    "dep_del15", "arr_del15",
    # duration / distance
    "crs_elapsed_min", "actual_elapsed_min", "air_time_min",
    "taxi_out_min", "taxi_in_min", "distance_mi",
    # status
    "is_cancelled", "cancellation_code", "cancellation_reason", "is_diverted",
    # delay attribution (populated by BTS only when arr_delay >= 15)
    "carrier_delay_min", "weather_delay_min", "nas_delay_min",
    "security_delay_min", "late_aircraft_delay_min", "has_delay_cause_detail",
    # BTS-only extras — NULL for 2004-2008
    "origin_airport_id", "dest_airport_id",
    "wheels_off_hhmm", "wheels_on_hhmm", "div_airport_landings",
    # lineage
    "_source_system", "_ingestion_timestamp", "_silver_processed_at", "_dq_warnings",
]
 
# The natural key used for dedup and for the surrogate flight_key.
NATURAL_KEY = ["flight_date", "carrier_code", "flight_number",
               "origin", "dest", "crs_dep_time_hhmm"]

In [0]:

def conform_historical(df):
    flight_date = F.make_date(num_int("Year"), num_int("Month"), num_int("DayofMonth"))
    crs_dep = hhmm("CRSDepTime")
    dep_delay = num_int("DepDelay")
    arr_delay = num_int("ArrDelay")
 
    out = (
        df.select(
            flight_date.alias("flight_date"),
            num_int("Year").alias("year"),
            num_int("Month").alias("month"),
            num_int("DayofMonth").alias("day_of_month"),
            num_int("DayOfWeek").alias("day_of_week"),
            F.upper(_clean_str("UniqueCarrier")).alias("carrier_code"),
            clean_tail("TailNum").alias("tail_number"),
            _clean_str("FlightNum").alias("flight_number"),  # already STRING in bronze
            F.upper(_clean_str("Origin")).alias("origin"),
            F.upper(_clean_str("Dest")).alias("dest"),
            crs_dep.alias("crs_dep_time_hhmm"),
            hhmm("DepTime").alias("dep_time_hhmm"),
            hhmm("CRSArrTime").alias("crs_arr_time_hhmm"),
            hhmm("ArrTime").alias("arr_time_hhmm"),
            dep_delay.alias("dep_delay_min"),
            arr_delay.alias("arr_delay_min"),
            num_int("CRSElapsedTime").alias("crs_elapsed_min"),
            num_int("ActualElapsedTime").alias("actual_elapsed_min"),
            num_int("AirTime").alias("air_time_min"),
            num_int("TaxiOut").alias("taxi_out_min"),
            num_int("TaxiIn").alias("taxi_in_min"),
            num_int("Distance").alias("distance_mi"),
            to_bool_01("Cancelled").alias("is_cancelled"),
            F.upper(_clean_str("CancellationCode")).alias("cancellation_code"),
            to_bool_01("Diverted").alias("is_diverted"),
            num_int("CarrierDelay").alias("carrier_delay_min"),
            num_int("WeatherDelay").alias("weather_delay_min"),
            num_int("NASDelay").alias("nas_delay_min"),
            num_int("SecurityDelay").alias("security_delay_min"),
            num_int("LateAircraftDelay").alias("late_aircraft_delay_min"),
            metadata_col(df, "_ingestion_timestamp", F.lit(None).cast("timestamp"))
                .alias("_ingestion_timestamp"),
        )
        # BTS-only columns have no historical equivalent.
        .withColumn("origin_airport_id", F.lit(None).cast("int"))
        .withColumn("dest_airport_id", F.lit(None).cast("int"))
        .withColumn("wheels_off_hhmm", F.lit(None).cast("int"))
        .withColumn("wheels_on_hhmm", F.lit(None).cast("int"))
        .withColumn("div_airport_landings", F.lit(None).cast("int"))
        .withColumn("_source_system", F.lit("asa_data_expo"))
    )
    return _add_derived(out)

In [0]:
def conform_recent(df):
    out = (
        df.select(
            F.to_date(F.col("FlightDate").cast("string")).alias("flight_date"),
            num_int("Year").alias("year"),
            num_int("Month").alias("month"),
            num_int("DayofMonth").alias("day_of_month"),
            num_int("DayOfWeek").alias("day_of_week"),
            F.upper(_clean_str("Reporting_Airline")).alias("carrier_code"),
            clean_tail("Tail_Number").alias("tail_number"),
            num_int("Flight_Number_Reporting_Airline").cast("string").alias("flight_number"),
            F.upper(_clean_str("Origin")).alias("origin"),
            F.upper(_clean_str("Dest")).alias("dest"),
            hhmm("CRSDepTime").alias("crs_dep_time_hhmm"),
            hhmm("DepTime").alias("dep_time_hhmm"),
            hhmm("CRSArrTime").alias("crs_arr_time_hhmm"),
            hhmm("ArrTime").alias("arr_time_hhmm"),
            num_int("DepDelay").alias("dep_delay_min"),
            num_int("ArrDelay").alias("arr_delay_min"),
            num_int("CRSElapsedTime").alias("crs_elapsed_min"),
            num_int("ActualElapsedTime").alias("actual_elapsed_min"),
            num_int("AirTime").alias("air_time_min"),
            num_int("TaxiOut").alias("taxi_out_min"),
            num_int("TaxiIn").alias("taxi_in_min"),
            num_int("Distance").alias("distance_mi"),
            to_bool_01("Cancelled").alias("is_cancelled"),
            F.upper(_clean_str("CancellationCode")).alias("cancellation_code"),
            to_bool_01("Diverted").alias("is_diverted"),
            num_int("CarrierDelay").alias("carrier_delay_min"),
            num_int("WeatherDelay").alias("weather_delay_min"),
            num_int("NASDelay").alias("nas_delay_min"),
            num_int("SecurityDelay").alias("security_delay_min"),
            num_int("LateAircraftDelay").alias("late_aircraft_delay_min"),
            opt(df, "OriginAirportID", num_int).alias("origin_airport_id"),
            opt(df, "DestAirportID", num_int).alias("dest_airport_id"),
            opt(df, "WheelsOff", hhmm).alias("wheels_off_hhmm"),
            opt(df, "WheelsOn", hhmm).alias("wheels_on_hhmm"),
            opt(df, "DivAirportLandings", num_int).alias("div_airport_landings"),
            metadata_col(df, "_ingestion_timestamp", F.lit(None).cast("timestamp"))
                .alias("_ingestion_timestamp"),
        )
        .withColumn("_source_system", F.lit("bts_transtats"))
    )
    return _add_derived(out)

In [0]:
def _add_derived(df):
    dep_delay, arr_delay = F.col("dep_delay_min"), F.col("arr_delay_min")
 
    return (
        df
        .withColumn("quarter", F.quarter("flight_date"))
        .withColumn("crs_dep_hour", hhmm_to_hour(F.col("crs_dep_time_hhmm")))
        .withColumn(
            "scheduled_departure_local_ts",
            local_timestamp(F.col("flight_date"), F.col("crs_dep_time_hhmm")),
        )
        # BTS convention: negative delay (early) is floored at 0 for "delay minutes".
        .withColumn("dep_delay_min_pos", F.greatest(dep_delay, F.lit(0)))
        .withColumn("arr_delay_min_pos", F.greatest(arr_delay, F.lit(0)))
        # THE harmonisation: derived from minutes on both sides, not mapped from source flags.
        .withColumn("dep_del15", F.when(dep_delay.isNotNull(), dep_delay >= 15))
        .withColumn("arr_del15", F.when(arr_delay.isNotNull(), arr_delay >= 15))
        .withColumn("cancellation_reason", cancellation_reason(F.col("cancellation_code")))
        .withColumn(
            "has_delay_cause_detail",
            F.coalesce(F.col("carrier_delay_min").isNotNull(), F.lit(False)),
        )
        .withColumn(
            "flight_key",
            F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("~"))
                                       for c in NATURAL_KEY]), 256),
        )
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

In [0]:
HARD_CHECKS = [
    (F.col("flight_date").isNull(), "MISSING_FLIGHT_DATE"),
    (F.col("carrier_code").isNull(), "MISSING_CARRIER"),
    (F.col("origin").isNull() | F.col("dest").isNull(), "MISSING_ROUTE"),
    (F.col("origin") == F.col("dest"), "ORIGIN_EQUALS_DEST"),
    (F.col("crs_dep_time_hhmm").isNull(), "MISSING_SCHEDULED_DEPARTURE"),
    (F.col("distance_mi").isNull() | (F.col("distance_mi") <= 0), "INVALID_DISTANCE"),
]
 
SOFT_CHECKS = [
    (~F.col("is_cancelled") & ~F.col("is_diverted") & F.col("arr_delay_min").isNull(),
     "COMPLETED_WITHOUT_ARRIVAL_DATA"),
    (F.col("is_cancelled") & F.col("arr_time_hhmm").isNotNull(), "CANCELLED_BUT_ARRIVED"),
    (F.abs(F.col("arr_delay_min")) > 2880, "IMPLAUSIBLE_ARR_DELAY"),      # > 48h
    (F.col("actual_elapsed_min") <= 0, "NON_POSITIVE_ELAPSED"),
    (F.col("air_time_min") > F.col("actual_elapsed_min"), "AIRTIME_EXCEEDS_ELAPSED"),
    (F.col("taxi_out_min") < 0, "NEGATIVE_TAXI"),
    (F.col("distance_mi") > 6000, "IMPLAUSIBLE_DISTANCE"),  # longest US domestic ~5100mi
    (F.abs(F.col("air_time_min")
           - (F.col("actual_elapsed_min") - F.col("taxi_out_min") - F.col("taxi_in_min"))) > 15,
     "AIRTIME_ELAPSED_MISMATCH")              
]

In [0]:
def _reason_array(checks):
    # F.when with no otherwise yields NULL when the condition is false OR null;
    # array_compact strips those out.
    return F.array_compact(F.array(*[F.when(cond, F.lit(code)) for cond, code in checks]))
 
 
def apply_dq(df):
    return (
        df
        .withColumn("_dq_errors", _reason_array(HARD_CHECKS))
        .withColumn("_dq_warnings", _reason_array(SOFT_CHECKS))
    )

In [0]:

COLUMN_DDL = """
    flight_key                    STRING    COMMENT 'SHA-256 of the natural key',
    flight_date                   DATE,
    year                          INT,
    quarter                       INT,
    month                         INT,
    day_of_month                  INT,
    day_of_week                   INT       COMMENT '1=Monday .. 7=Sunday',
    carrier_code                  STRING    COMMENT 'UniqueCarrier (pre-2009) / Reporting_Airline',
    tail_number                   STRING,
    flight_number                 STRING,
    origin                        STRING,
    dest                          STRING,
    crs_dep_time_hhmm             INT       COMMENT 'Local hhmm; 2400 normalised to 0',
    dep_time_hhmm                 INT,
    crs_arr_time_hhmm             INT,
    arr_time_hhmm                 INT,
    crs_dep_hour                  INT,
    scheduled_departure_local_ts  TIMESTAMP COMMENT 'LOCAL wall-clock at origin, not UTC',
    dep_delay_min                 INT       COMMENT 'Signed; negative = early',
    arr_delay_min                 INT       COMMENT 'Signed; negative = early',
    dep_delay_min_pos             INT       COMMENT 'Floored at 0, BTS convention',
    arr_delay_min_pos             INT,
    dep_del15                     BOOLEAN   COMMENT 'Derived: dep_delay_min >= 15',
    arr_del15                     BOOLEAN   COMMENT 'Derived: arr_delay_min >= 15',
    crs_elapsed_min               INT,
    actual_elapsed_min            INT,
    air_time_min                  INT,
    taxi_out_min                  INT,
    taxi_in_min                   INT,
    distance_mi                   INT,
    is_cancelled                  BOOLEAN,
    cancellation_code             STRING    COMMENT 'A/B/C/D',
    cancellation_reason           STRING,
    is_diverted                   BOOLEAN,
    carrier_delay_min             INT       COMMENT 'Populated only when arr_delay >= 15',
    weather_delay_min             INT,
    nas_delay_min                 INT,
    security_delay_min            INT,
    late_aircraft_delay_min       INT,
    has_delay_cause_detail        BOOLEAN,
    origin_airport_id             INT       COMMENT 'BTS only; NULL pre-2009',
    dest_airport_id               INT       COMMENT 'BTS only; NULL pre-2009',
    wheels_off_hhmm               INT       COMMENT 'BTS only; NULL pre-2009',
    wheels_on_hhmm                INT       COMMENT 'BTS only; NULL pre-2009',
    div_airport_landings          INT       COMMENT 'BTS only; NULL pre-2009',
    _source_system                STRING,
    _ingestion_timestamp          TIMESTAMP,
    _silver_processed_at          TIMESTAMP,
    _dq_warnings                  ARRAY<STRING> COMMENT 'Soft anomalies; row still usable'
"""

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER} (
{COLUMN_DDL}
)
USING DELTA
CLUSTER BY (flight_date, origin)
COMMENT 'Conformed flight legs, 2004-present. Union of ASA Data Expo and BTS TranStats.'
""")
 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE} (
{COLUMN_DDL},
    _dq_errors                    ARRAY<STRING> COMMENT 'Hard failures; why the row was rejected'
)
USING DELTA
CLUSTER BY (flight_date, _source_system)
COMMENT 'Rows rejected from silver.flights, retained with reason codes.'
""")
 
print("tables ready")

In [0]:
def build_year(year, source):
    """Conform, dedup, split and write a single year. Returns an audit dict."""
    _t0 = time.time()
    if source == "historical":
        bronze, conform = spark.table(BRONZE_HIST), conform_historical
    elif source == "recent":
        bronze, conform = spark.table(BRONZE_RECENT), conform_recent
    else:
        raise ValueError(f"unknown source: {source}")
 
    raw = bronze.filter(F.col("Year").cast("int") == year)
    raw_count = raw.count()
    if raw_count == 0:
        print(f"  [{year}] no rows in bronze — skipping")
        return None
 
    conformed = apply_dq(conform(raw))
 
    # Defensive dedup. Historical was deduped in Bronze; recent was not, and a
    # re-uploaded BTS month would show up here.
    win = W.partitionBy("flight_key").orderBy(F.col("_ingestion_timestamp").desc_nulls_last())
    deduped = (
        conformed.withColumn("_rn", F.row_number().over(win))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    (deduped.filter(F.size("_dq_errors") == 0)
        .select(*SILVER_COLUMNS)
        .write.format("delta").mode("overwrite")
        .option("replaceWhere", f"year = {year}")
        .saveAsTable(SILVER))

    (deduped.filter(F.size("_dq_errors") > 0)
        .select(*SILVER_COLUMNS, "_dq_errors")
        .write.format("delta").mode("overwrite")
        .option("replaceWhere", f"year = {year}")
        .saveAsTable(QUARANTINE))

    clean_count = spark.table(SILVER).filter(F.col("year") == year).count()
    reject_count = spark.table(QUARANTINE).filter(F.col("year") == year).count()
    dedup_count = clean_count + reject_count
 
 
    audit = {
        "year": year, "source": source, "bronze_rows": raw_count,
        "seconds": round(time.time() - _t0, 1),
        "after_dedup": dedup_count, "dupes_removed": raw_count - dedup_count,
        "silver_rows": clean_count, "quarantined": reject_count,
    }
    print(f"  [{year}] bronze={raw_count:,}  dupes={audit['dupes_removed']:,}  "
          f"silver={clean_count:,}  quarantine={reject_count:,}")
    return audit

In [0]:
audits = []
for source, years in PILOT_YEARS.items():
    print(f"--- {source} ---")
    for y in years:
        result = build_year(y, source)
        if result:
            audits.append(result)
 
display(spark.createDataFrame(audits))

In [0]:
spark.sql(f"OPTIMIZE {SILVER}")

In [0]:
display(spark.createDataFrame(audits).select(
    "year", "source", "bronze_rows", "dupes_removed", "silver_rows", "quarantined",
    (F.col("silver_rows") + F.col("quarantined") + F.col("dupes_removed")
     - F.col("bronze_rows")).alias("unexplained_gap"),
))

In [0]:
display(
    spark.table(QUARANTINE)
    .select("year", "_source_system", F.explode("_dq_errors").alias("reason"))
    .groupBy("year", "_source_system", "reason")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    spark.table(SILVER).groupBy("year", "_source_system").agg(
        F.count("*").alias("flights"),
        F.round(F.avg(F.col("arr_del15").cast("int")) * 100, 2).alias("pct_arr_delayed_15"),
        F.round(F.avg(F.col("dep_del15").cast("int")) * 100, 2).alias("pct_dep_delayed_15"),
        F.round(F.avg(F.col("is_cancelled").cast("int")) * 100, 2).alias("pct_cancelled"),
        F.round(F.avg(F.col("is_diverted").cast("int")) * 100, 2).alias("pct_diverted"),
        F.round(F.avg("arr_delay_min"), 2).alias("avg_arr_delay_min"),
        F.round(F.avg("distance_mi"), 1).alias("avg_distance_mi"),
        F.countDistinct("carrier_code").alias("carriers"),
        F.countDistinct("origin").alias("origin_airports"),
    ).orderBy("year"))

In [0]:
display(
    spark.table(SILVER)
    .select("year", F.explode("_dq_warnings").alias("warning"))
    .groupBy("year", "warning")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
bts_only = ["origin_airport_id", "dest_airport_id", "wheels_off_hhmm",
            "wheels_on_hhmm", "div_airport_landings"]
display(
    spark.table(SILVER).groupBy("year").agg(*[
        F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(f"pct_null_{c}")
        for c in bts_only
    ]).orderBy("year")
)

In [0]:
display(
    spark.table(SILVER)
    .filter(F.col("arr_delay_min").isNotNull())
    .select("year", "flight_date", "carrier_code", "origin", "dest",
            "crs_dep_time_hhmm", "scheduled_departure_local_ts",
            "dep_delay_min", "dep_del15", "arr_delay_min", "arr_del15",
            "cancellation_code", "cancellation_reason")
    .sample(0.0001)
    .limit(20)
)

In [0]:
%sql
SELECT month, COUNT(*) FROM airline_analytics.silver.flights
WHERE year = 2026 GROUP BY month ORDER BY month

In [0]:
%sql
SELECT COUNT(*) FROM airline_analytics.silver.flights